# Energy balance evaluation
---
*J. Michelle Hu  
University of Utah  
March 2025*  


In [ ]:
from pathlib import PurePath
import sys
import os
import xarray as xr

import matplotlib.pyplot as plt
import pyproj

import pandas as pd

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

## Directory

In [ ]:
basin = 'animas'
WY = 2021
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
basindirs = h.fn_list(workdir, f'{basin}*isnobal/wy{WY}/*/')
basindirs

## Specify a point for time series evaluation

In [ ]:
snotel_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

# Basin polygon file
poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]

# SNOTEL all sites geojson fn
allsites_fn = h.fn_list(snotel_dir, 'snotel_sites_32613.json')[0]

# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=allsites_fn, buffer=200)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

ST_arr = ['CO'] * len(sitenums)
gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=int(WY), return_meta=True)

## Plot the simulated snow depth for these locations over the water year


In [ ]:
if basin == 'yampa' and WY == 2021:
    preciptest = True
else:
    preciptest = False

In [ ]:
# Read in iSnobal output
ds_dict = dict()
thisvar = 'thickness'
if preciptest:
    labels = ['iSnobal-HRRR', 'HRRR-MODIS', 'HRRR-MODIS-preciptest'] # to follow csv output extract nomenclature
else:
    labels = ['iSnobal-HRRR', 'HRRR-MODIS'] # to follow csv output extract nomenclature

for kdx, label in enumerate(labels):
    model_ts_fn = f'{PurePath(basindirs[0]).parents[0].as_posix()}/{basin}_{label}_{thisvar}_snotelmetloom_wy{WY}.csv'
    print(model_ts_fn)

    df = pd.read_csv(model_ts_fn, index_col=0)
    df.index.name = 'Date'
    # Set as DatetimeIndex
    df.index = pd.to_datetime(df.index)
    if label == 'iSnobal-HRRR':
        # change to 'Baseline' for dictionary storage
        label = 'Baseline'
    ds_dict[f'{label}_{thisvar}'] = df

In [ ]:
import seaborn as sns
sns.set_palette("icefire")

In [ ]:
snowvar_col = 'SNOWDEPTH_m'
# snotel_df, gdf, sitenum, sitename = proc.get_snotel(snotel_dir=snotel_dir, sitenums=sitenums, WY=WY)
linestyle = '-' #':'
linewidth = 0.5
marker = '.'
color = 'k'
colors = ['navy', 'dodgerblue']
fig, axes = plt.subplots(len(sitenames), 1, figsize=(12, 2 * len(sitenames)), sharex=True, sharey=True)
# Plot by site
for jdx, sitename in enumerate(sitenames):
    ax = axes[jdx]
    # Extract dfs
    snotel_df = snotel_dfs[sitename]
    baseline_df = ds_dict['Baseline_thickness'][sitename]
    hrrrmodis_df = ds_dict['HRRR-MODIS_thickness'][sitename]
    if preciptest:
       preciptest_df = ds_dict['HRRR-MODIS-preciptest_thickness'][sitename]
    (snotel_df[snowvar_col]).plot(ax=ax,
                                  label=f'{sitename} Snow Depth [m]',
                                  linestyle=linestyle,
                                  linewidth=linewidth,
                                  color='gray',
                                  marker=marker,
                                  markersize=4,
                                  )
    baseline_df.plot(ax=ax,
                     label='Baseline Snow Depth [m]',
                     linestyle=linestyle,
                     linewidth=1.2,
                     )
    hrrrmodis_df.plot(ax=ax,
                      label='HRRR-MODIS Snow Depth [m]',
                    #   linestyle='--',
                      linewidth=1.2,
                      )
    if preciptest:
       preciptest_df.plot(ax=ax,
                        label='HRRR-MODIS preciptest Snow Depth [m]',
                        color='mediumpurple',
                        marker='|',
                        markersize=3,
                        linestyle=linestyle,
                        linewidth=0,
                        )
    ax.set_ylim(0, 3)
    ax.set_xlim(snotel_df.index[0], snotel_df.index[-1])
    # Put legend outside of plot
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # Add a grid with very light grey lines on both major and minor axes
    ax.grid(color='lightgrey', linewidth=0.5)
plt.suptitle(f'{basin} WY {WY} SNOTEL Snow Depth')
plt.tight_layout();

In [ ]:
def trim_datasets(ds1, ds2):
        # Trim the datasets to the same time period
        # if HRRR-MODIS ends earlier than Baseline, trim Baseline time period
        if ds1.time.values[-1] > ds2.time.values[-1]:
            ds1 = ds1.sel(time=slice(ds1.time.values[0], ds2.time.values[-1]))
        # if HRRR-MODIS ends later than Baseline, trim HRRR-MODIS time period
        elif ds2.time.values[-1] > ds1.time.values[-1]:
            ds2 = ds2.sel(time=slice(ds2.time.values[0], ds1.time.values[-1]))
        # if HRRR-MODIS starts later than Baseline, trim Baseline time period
        if ds1.time.values[0] < ds2.time.values[0]:
            ds1 = ds1.sel(time=slice(ds2.time.values[0], ds1.time.values[-1]))
        # if HRRR-MODIS starts earlier than Baseline, trim HRRR-MODIS time period
        elif ds2.time.values[0] < ds1.time.values[0]:
            ds2 = ds2.sel(time=slice(ds2.time.values[0], ds1.time.values[-1]))
        return ds1, ds2

### Load em data

In [ ]:
# animas_HRRR-MODIS_em_snotelmetloom_wy2021.nc
# Read in em output
em_dict = dict()
thisvar = 'em'
for kdx, label in enumerate(labels):
    model_ts_fn = f'{PurePath(basindirs[0]).parents[0].as_posix()}/{basin}_{label}_{thisvar}_snotelmetloom_wy{WY}.nc'
    print(model_ts_fn)
    ds = xr.open_dataset(model_ts_fn)
    if label == 'iSnobal-HRRR':
        # change to 'Baseline' for dictionary storage
        label = 'Baseline'
    em_dict[f'{label}'] = ds

In [ ]:
# em_ds_list = [xr.open_mfdataset(h.fn_list(basindir, '*/em.nc'), drop_variables=['projection', 'snowmelt', 'SWI', 'cold_content']) for basindir in basindirs]

In [ ]:
# for f in em_ds_list:
#     print(len(f.time))

In [ ]:
# em_ds_list[0], em_ds_list[1] = trim_datasets(em_ds_list[0], em_ds_list[1])
# for f in em_ds_list:
#     print(len(f.time))

In [ ]:
# em_var = 'net_rad'
# em_dict = dict()
# labels = ['Baseline', 'HRRR-MODIS']

# for kdx, em_ds in enumerate(em_ds_list):
#     label = labels[kdx]
#     # Extract values at snotel coordinates
#     em_data = em_ds[em_var].sel(x=list(gdf_metloom.geometry.x.values),
#                                     y=list(gdf_metloom.geometry.y.values),
#                                     method='nearest')
#     # Convert to a dict
#     for jdx, sitename in enumerate(sitenames):
#         ds = em_data[:, jdx, jdx]
#         em_dict[f'{sitename}_{label}'] = ds.values

# # Turn it into a dataframe
# em_datadf = pd.DataFrame(em_dict, index=ds['time'].values)

In [ ]:
# data_dict = dict()

# for kdx, em_ds in enumerate(em_ds_list):
#     label = labels[kdx]
#     # Extract values at snotel coordinates
#     em_data = em_ds.sel(x=list(gdf_metloom.geometry.x.values),
#                                     y=list(gdf_metloom.geometry.y.values),
#                                     method='nearest')
#     data_dict[label] = em_data.load()

In [ ]:
em_dict

In [ ]:
fixednames = [sitename.replace(' ', '_').replace('(', '').replace(')', '') for sitename in sitenames]
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/hrrrmodis_checks'
overwrite = False
plotanyways = True

### Plot the EB terms!

In [ ]:
# fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
data_vars = em_dict['Baseline'].data_vars
for jdx, sitename in enumerate(sitenames):
    print(sitename)
    fixedname = fixednames[jdx]
    outname = f'{outdir}/{basin}_snotel_{fixedname}_em_{WY}.png'
    fig, axes = plt.subplots(len(data_vars), 1, figsize=(10,1.2 * len(data_vars)), sharex=True, sharey=True)
    for kdx, data_var in enumerate(data_vars):
        print(data_var, kdx)
        ax = axes[kdx]
        # Plot the var for the site
        em_dict['Baseline'][data_var][:, jdx, jdx].plot(ax=ax, label='Baseline', color='r', marker='.', markersize=2, linewidth=0)
        em_dict['HRRR-MODIS'][data_var][:, jdx, jdx].plot(ax=ax, label='HRRR-MODIS', color='royalblue', linewidth=1, linestyle='-')
        ax.annotate(data_var, xy=(0.01, 1.05), xycoords='axes fraction')
        ax.set_ylim(-50, 100)
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        # Add more ticks
        ax.minorticks_on()
        ax.grid(color='lightgrey', linewidth=0.3, which='both')
    plt.suptitle(f'{basin} WY {WY} SNOTEL site {sitename}')
    plt.tight_layout()
    if os.path.exists(outname) and not overwrite:
        print(f'File exists: {outname}')
    else:
        plt.savefig(outname, dpi=300, bbox_inches='tight')
        plt.close()

### Load net solar files

In [ ]:
# net solar and albedo are located in smrf_energy_balance_YYYYMMDD.nc for baseline
# net solar and albedo are located in net_solar.nc for HRRR-MODIS
# netsolar_ds_list = [xr.open_mfdataset(h.fn_list(basindirs[0], '*/*energy_balance*.nc'),
#                                       drop_variables=['albedo_ir', 'albedo_vis'])] + [xr.open_mfdataset(h.fn_list(basindirs[1], '*/net_solar.nc'),
#                                                                                                         drop_variables=['albedo', 'DSWRF', 'illumination_angle'])]

netsolar_ds_list = [xr.open_mfdataset(f'{basindirs[0]}*/*energy_balance*.nc')] + [xr.open_mfdataset(f'{basindirs[1]}*/net_solar.nc', drop_variables=['illumination_angle'])]

In [ ]:
# Resample hourly to daily timesteps
for kdx, ds in enumerate(netsolar_ds_list):
    netsolar_ds_list[kdx] = ds.resample(time='1D').mean().load()

In [ ]:
for f in netsolar_ds_list:
    print(len(f.time))

In [ ]:
# # Trim the datasets to the same time period
# netsolar_ds_list[0], netsolar_ds_list[1] = trim_datasets(netsolar_ds_list[0], netsolar_ds_list[1])

# Net solar

In [ ]:
em_var = 'net_solar'
em_ds_list = netsolar_ds_list
em_dict = dict()
for kdx, em_ds in enumerate(em_ds_list):
    label = labels[kdx]
    # Extract values at snotel coordinates
    em_data = em_ds[em_var].sel(x=list(gdf_metloom.geometry.x.values),
                                    y=list(gdf_metloom.geometry.y.values),
                                    method='nearest')
    em_dict[label] = em_data.load()
    # # Convert to a dict
    # for jdx, sitename in enumerate(sitenames):
    #     ds = em_data[:, jdx, jdx]
    #     em_dict[f'{sitename}_{label}'] = ds.values

# Pull the DSWRF variable from the HRRR-MODIS dataset as well
em_ds = netsolar_ds_list[1]
em_data = em_ds['DSWRF'].sel(x=list(gdf_metloom.geometry.x.values),
                            y=list(gdf_metloom.geometry.y.values),
                            method='nearest')
em_dict['DSWRF'] = em_data.load()
# for jdx, sitename in enumerate(sitenames):
#     ds = em_data[:, jdx, jdx]
#     em_dict[f'{sitename}_DSWRF'] = ds.values

# # Turn it into a dataframe
# net_solar_datadf = pd.DataFrame(em_dict, index=ds['time'].values)

In [ ]:
data_dict = em_dict
data_vars = em_dict['Baseline'].data_vars
# Plot site by site
# fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
for jdx, sitename in enumerate(sitenames):
    # ax = axes[jdx]
    # site_df = datadf[[col for col in datadf.columns if sitename in col]]
    # # Plot baseline
    # site_df[[col for col in site_df.columns if 'Baseline' in col]].plot(ax=ax, linewidth=1.5, color='gold')
    # # Plot HRRR-MODIS
    # site_df[[col for col in site_df.columns if 'HRRR' in col]].plot(ax=ax, linewidth=0.75, linestyle='--', color='firebrick')
    # # Plot DSWRF
    # site_df[[col for col in site_df.columns if 'DSWRF' in col]].plot(ax=ax, linewidth=0.5, color='gray')
    # # Stop at July
    # # ax.set_xlim(pd.Timestamp('2020-10-01'), pd.Timestamp('2021-06-30'))
    # ax.set_ylim(-50, 550)
    # # Put legend outside of plot
    # ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    print(sitename)
    fig, axes = plt.subplots(len(data_vars), 1, figsize=(12,1.5 * len(data_vars)), sharex=True, sharey=True)
    for kdx, data_var in enumerate(data_vars):
        ax = axes[kdx]
        # Plot the var for the site
        data_dict['Baseline'][data_var][:, jdx, jdx].plot(ax=ax, label='Baseline', color='gold', linewidth=0)
        data_dict['HRRR-MODIS'][data_var][:, jdx, jdx].plot(ax=ax, label='HRRR-MODIS', color='firebrick', linewidth=1, linestyle='-')
        data_dict['HRRR-MODIS-preciptest'][data_var][:, jdx, jdx].plot(ax=ax, label='HRRR-MODIS-preciptest', color='mediumpurple', marker='|', markersize=3, linewidth=0)
        # Plot DSWRF
        data_dict['DSWRF'][:, jdx, jdx].plot(ax=ax, label='DSWRF', color='gray', linewidth=0.5)
        ax.annotate(data_var, xy=(0.01, 1.05), xycoords='axes fraction')
        # ax.set_ylim(-50, 100)
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        # Add more ticks
        ax.minorticks_on()
        ax.grid(color='lightgrey', linewidth=0.3, which='both')
    plt.suptitle(f'{basin} WY {WY} SNOTEL site {sitename}')
    plt.tight_layout()
# plt.suptitle(f'{basin} WY {WY} SNOTEL site [{em_var}]')
# plt.tight_layout()
# outname = f'{outdir}/{basin}_snotel_{em_var}_dswrf_{WY}.png'
# if os.path.exists(outname) and not overwrite:
#     print(f'File exists: {outname}')
# else:
#     plt.savefig(outname, dpi=300, bbox_inches='tight')

In [ ]:
# datadf = net_solar_datadf
# # Plot site by site
# fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
# for jdx, sitename in enumerate(sitenames):
#     ax = axes[jdx]
#     site_df = datadf[[col for col in datadf.columns if sitename in col]]
#     # Plot baseline
#     site_df[[col for col in site_df.columns if 'Baseline' in col]].plot(ax=ax, linewidth=1.5, color='gold')
#     # Plot HRRR-MODIS
#     site_df[[col for col in site_df.columns if 'HRRR' in col]].plot(ax=ax, linewidth=0.75, linestyle='--', color='firebrick')
#     # Stop at July
#     # ax.set_xlim(pd.Timestamp('2020-10-01'), pd.Timestamp('2021-06-30'))
#     ax.set_ylim(-50, 200)
#     # Put legend outside of plot
#     ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
# plt.suptitle(f'{basin} WY {WY} SNOTEL site [{em_var}]')
# plt.tight_layout()
# plt.savefig(f'{outdir}/{basin}_snotel_{em_var}_{WY}.png', dpi=300, bbox_inches='tight')

In [ ]:
# datadf = net_solar_datadf
# # Plot site by site
# fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
# for jdx, sitename in enumerate(sitenames):
#     ax = axes[jdx]
#     site_df = datadf[[col for col in datadf.columns if sitename in col]]
#     # Plot baseline
#     site_df[[col for col in site_df.columns if 'Baseline' in col]].plot(ax=ax, linewidth=1.5, color='gold')
#     # Plot HRRR-MODIS
#     site_df[[col for col in site_df.columns if 'HRRR' in col]].plot(ax=ax, linewidth=0.75, linestyle='--', color='firebrick')
#     # Plot DSWRF
#     site_df[[col for col in site_df.columns if 'DSWRF' in col]].plot(ax=ax, linewidth=0.5, color='gray')
#     # Stop at July
#     # ax.set_xlim(pd.Timestamp('2020-10-01'), pd.Timestamp('2021-06-30'))
#     ax.set_ylim(-50, 550)
#     # Put legend outside of plot
#     ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
# plt.suptitle(f'{basin} WY {WY} SNOTEL site [{em_var}]')
# plt.tight_layout()
# outname = f'{outdir}/{basin}_snotel_{em_var}_dswrf_{WY}.png'
# if os.path.exists(outname) and not overwrite:
#     print(f'File exists: {outname}')
# else:
#     plt.savefig(outname, dpi=300, bbox_inches='tight')

In [ ]:
# Free up some memory
del em_var, em_dict, em_ds, em_ds_list, em_data

# Albedo

In [ ]:
[f for f in netsolar_ds_list[0].data_vars if 'albedo' in f]
# [f for f in netsolar_ds_list[1].data_vars if 'albedo' in f]

In [ ]:
em_var = 'albedo'
em_ds_list = netsolar_ds_list
em_dict = dict()
for kdx, em_ds in enumerate(em_ds_list):
    label = labels[kdx]
    data_varnames = [f for f in em_ds_list[kdx].data_vars if em_var in f]
    for data_var in data_varnames:
        # Extract values at snotel coordinates
        em_data = em_ds[data_var].sel(x=list(gdf_metloom.geometry.x.values),
                                        y=list(gdf_metloom.geometry.y.values),
                                        method='nearest')
        # Convert to a dict
        for jdx, sitename in enumerate(sitenames):
            ds = em_data[:, jdx, jdx]
            em_dict[f'{sitename}_{label}_{data_var}'] = ds.values

# Turn it into a dataframe
datadf = pd.DataFrame(em_dict, index=ds['time'].values)

In [ ]:
# Plot site by site
fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
for jdx, sitename in enumerate(sitenames):
    ax = axes[jdx]
    site_df = datadf[[col for col in datadf.columns if sitename in col]]
    # Plot baseline albedo vis
    site_df[[col for col in site_df.columns if '_vis' in col]].plot(ax=ax, linewidth=0.75)
    # Plot baseline albedo IR
    site_df[[col for col in site_df.columns if '_ir' in col]].plot(ax=ax, linewidth=0.75)
    # Plot HRRR-MODIS and adjust for albedo units
    (site_df[[col for col in site_df.columns if 'HRRR' in col]] / 10000).plot(ax=ax, linewidth=0.75, linestyle='--')
    ax.set_ylim(0, 1)
    # Put legend outside of plot
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.suptitle(f'{basin} WY {WY} SNOTEL site [{em_var}]')
plt.tight_layout()
outname = f'{outdir}/{basin}_snotel_solar_{WY}.png'
if os.path.exists(outname) and not overwrite:
    print(f'File exists: {outname}')
else:
    plt.savefig(outname, dpi=300, bbox_inches='tight')

### Plot albedo on top of DSWRF and net solar for HRRR-MODIS

In [ ]:
# Plot site by site
fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
for jdx, sitename in enumerate(sitenames):
    ax = axes[jdx]
    # Get the solar bits
    site_df = net_solar_datadf[[col for col in net_solar_datadf.columns if sitename in col]]
    # Plot baseline
    site_df[[col for col in site_df.columns if 'Baseline' in col]].plot(ax=ax, linewidth=1.5, color='gold')
    # Plot HRRR-MODIS
    site_df[[col for col in site_df.columns if 'HRRR' in col]].plot(ax=ax, linewidth=0.75, linestyle='--', color='firebrick')
    # # Plot DSWRF
    # site_df[[col for col in site_df.columns if 'DSWRF' in col]].plot(ax=ax, linewidth=0.5, color='gray')
    # ax.legend(loc='center left', bbox_to_anchor=(1.1, 0.7))
    # ax.set_ylim(-50, 550)
    ax.legend(loc='center left', bbox_to_anchor=(1.1, 0.6))
    # ax.set_ylim(-50, 250)
    # Turn the yaxis into a log scale
    ax.set_yscale('log')

    # Now plot the albedo
    ax2 = ax.twinx()
    site_df = datadf[[col for col in datadf.columns if sitename in col]]
    # Plot HRRR-MODIS and adjust for albedo units
    (site_df[[col for col in site_df.columns if 'HRRR' in col]] / 10000).plot(ax=ax2, linewidth=0.75, linestyle='--', color = 'k')
    ax2.set_ylim(0, 1)
    # Put legend outside of plot
    ax2.legend(loc='center left', bbox_to_anchor=(1.1, 0.275))
plt.suptitle(f'{basin} WY {WY} SNOTEL site [{em_var}]')
plt.tight_layout()
outname = f'{outdir}/{basin}_snotel_albedo_solar_{WY}.png'
if os.path.exists(outname) and not overwrite:
    print(f'File exists: {outname}')
else:
    plt.savefig(outname, dpi=300, bbox_inches='tight')

# DSWRF

In [ ]:
# Check that DSWRF in net_solar.nc is identical to DSWRF  in hrrr.yyyymmdd.dswrf.nc

# Check out surface temperatures

In [ ]:
# Load the snow.nc files to check out temperature
snow_ds_list = [xr.open_mfdataset(h.fn_list(basindir, '*/snow.nc'), drop_variables=['projection', 'thickness', 'snow_density', 'specific_mass', 'liquid_water', 'thickness_lower', 'water_saturation']) for basindir in basindirs]

In [ ]:
for f in snow_ds_list:
    print(len(f.time))

In [ ]:
big_snow_dict = dict()
labels = ['Baseline', 'HRRR-MODIS', 'HRRR-MODIS-preciptest']

for kdx, ds in enumerate(snow_ds_list):
    label = labels[kdx]
    # Extract values at snotel coordinates
    snow_data = ds.sel(x=list(gdf_metloom.geometry.x.values),
                        y=list(gdf_metloom.geometry.y.values),
                        method='nearest')
    big_snow_dict[label] = snow_data.load()

In [ ]:
# Drop the projection

In [ ]:
# fig, axes = plt.subplots(len(sitenames), 1, figsize=(12,1.5 * len(sitenames)), sharex=True, sharey=True)
data_vars = big_snow_dict['Baseline'].data_vars
for jdx, sitename in enumerate(sitenames):
    print(sitename)
    fig, axes = plt.subplots(len(data_vars), 1, figsize=(12,1.5 * len(data_vars)), sharex=True, sharey=True)
    for kdx, data_var in enumerate(data_vars):
        ax = axes[kdx]
        # Plot the var for the site
        # big_em_dict['Baseline'][data_var][:, jdx, jdx].plot(ax=ax, label='Baseline', linewidth=1.5)
        big_snow_dict['Baseline'][data_var][:, jdx, jdx].plot(ax=ax, label='Baseline', color='r', marker='.', markersize=2, linewidth=0)
        big_snow_dict['HRRR-MODIS'][data_var][:, jdx, jdx].plot(ax=ax, label='HRRR-MODIS', color='royalblue', linewidth=1, linestyle='-')
        big_snow_dict['HRRR-MODIS-preciptest'][data_var][:, jdx, jdx].plot(ax=ax, label='HRRR-MODIS-preciptest', color='mediumpurple', marker='|', markersize=3, linewidth=0)
        ax.annotate(data_var, xy=(0.01, 1.05), xycoords='axes fraction')
        # ax.set_ylim(-50, 100)
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        # Add more ticks
        ax.minorticks_on()
        ax.grid(color='lightgrey', linewidth=0.3, which='both')
    plt.suptitle(f'{basin} WY {WY} SNOTEL site {sitename}')
    plt.tight_layout()

# Need to go back and overhaul conversion to dataframe, no real need if we stick to dictionaries!